# DS4 Step 2, Path C' -- Network-focused Phenotyping

**Notebook:** `132_ds4_network_focused_phenotype.ipynb`  
**Plan:** `outputs/reports/stage4/step2_phenotyping/ds4_step2_exploration_plan.md` SS3

---

## Question

Does DS4's network structure (NeT) form its own independent phenotype axis?

## Redundancy finding that shaped this notebook's feature choice

NeT's own richer feature set (`rho_self`, `HHI_net`, `H_trans`) is severely internally redundant: `corr(rho_self,HHI_net)=0.867`, `corr(rho_self,H_trans)=-0.950`, `corr(HHI_net,H_trans)=-0.919` -- close to mechanical (high self-transition almost necessarily means high concentration and low transition entropy). Using all three would triple-count one signal, not add real dimensions. Only `I_consec` (the InT-NeT mutual-information bridge) is close to independent of the other three. **Features used here: `rho_self` + `I_consec`** -- the two non-redundant NeT-side quantities, reproduced live below.


## FLAG decisions

| Flag | Decision |
|---|---|
| FLAG 1 | `rho_self` + `I_consec` only, not the full redundant NeT feature set |
| FLAG 2 | Same complete-case population as Phase 1 (N=641) |
| FLAG 3 | Stability-first k-selection |


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

OUT_DIR = Path("outputs/model_outputs/ds4_phenotyping")
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

FEATURES = ["rho_self", "I_consec"]
RANDOM_STATE = 42


In [ ]:
stop = pd.read_csv("outputs/model_outputs/ds4_stochastic_processes/participant_parameters.csv")
stm = pd.read_csv("outputs/model_outputs/ds4_statistical_mechanics/participant_ds4_stm_parameters.csv")
net = pd.read_csv("outputs/model_outputs/ds4_network_models/participant_network_metrics.csv")
phase1 = pd.read_csv(OUT_DIR / "ds4_cluster_assignments.csv").rename(
    columns={"cluster": "phase1_cluster"})

df = stop[["participant_id_canonical", "fit_valid"]].rename(columns={"fit_valid": "stop_valid"})
df = df.merge(stm[["participant_id_canonical", "fit_valid"]].rename(
    columns={"fit_valid": "stm_valid"}), on="participant_id_canonical", how="left")
df = df.merge(net[["participant_id_canonical", "rho_self", "I_consec", "HHI_net",
    "H_trans", "eligible"]].rename(columns={"eligible": "net_eligible"}),
    on="participant_id_canonical", how="left")
df = df.merge(phase1, on="participant_id_canonical", how="left")

complete_mask = (
    df["stop_valid"].fillna(False) & df["stm_valid"].fillna(False) &
    df["net_eligible"].fillna(False) & df[FEATURES].notna().all(axis=1)
)
complete = df[complete_mask].reset_index(drop=True)
print(f"Complete-case N: {len(complete)} (same population as Phase 1)")


In [ ]:
print("Full NeT feature redundancy check (live reproduction):")
print(complete[["rho_self", "HHI_net", "H_trans", "I_consec"]].corr().round(3))
print(f"\nUsed features ({FEATURES}) correlation: "
      f"{complete[FEATURES[0]].corr(complete[FEATURES[1]]):.4f}")

X = StandardScaler().fit_transform(complete[FEATURES])


In [ ]:
def jaccard_cluster_stability(X, k, n_boot=100, random_state=RANDOM_STATE):
    rng = np.random.RandomState(random_state)
    n = X.shape[0]
    ref = KMeans(n_clusters=k, n_init=10, random_state=random_state).fit(X)
    ref_labels = ref.labels_
    jaccards = []
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        seed = int(rng.randint(0, 1_000_000))
        boot = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(X[idx])
        boot_on_full = boot.predict(X)
        best_js = []
        for c in range(k):
            ref_set = set(np.where(ref_labels == c)[0])
            best_j = 0.0
            for c2 in range(k):
                boot_set = set(np.where(boot_on_full == c2)[0])
                union = len(ref_set | boot_set)
                if union == 0:
                    continue
                best_j = max(best_j, len(ref_set & boot_set) / union)
            best_js.append(best_j)
        jaccards.append(np.mean(best_js))
    return np.mean(jaccards)


k_candidates = range(2, 7)
sil_scores, jaccard_by_k, sizes_by_k = {}, {}, {}
for k in k_candidates:
    labels = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit_predict(X)
    sil_scores[k] = silhouette_score(X, labels)
    jaccard_by_k[k] = jaccard_cluster_stability(X, k)
    sizes_by_k[k] = pd.Series(labels).value_counts().sort_index().tolist()
    verdict = "STABLE" if jaccard_by_k[k] >= 0.75 else "not stable"
    print(f"k={k}: silhouette={sil_scores[k]:.4f}, jaccard={jaccard_by_k[k]:.4f} "
          f"({verdict}), sizes={sizes_by_k[k]}")

MIN_CELL = 10
stable_ks = [k for k in k_candidates
             if jaccard_by_k[k] >= 0.75 and min(sizes_by_k[k]) >= MIN_CELL]
if stable_ks:
    K_CHOSEN = max(stable_ks, key=lambda k: sil_scores[k])
    print(f"\nChosen k={K_CHOSEN} (highest silhouette among STABLE candidates "
          f"with all clusters >= {MIN_CELL})")
else:
    K_CHOSEN = max(sil_scores, key=sil_scores.get)
    print(f"\nNO FEASIBLE k -- no candidate was both bootstrap-stable and free of "
          f"cells below {MIN_CELL}. Falling back to silhouette-argmax k={K_CHOSEN}; "
          f"flag this solution as UNVALIDATED in the writeup.")


In [ ]:
labels_final = KMeans(n_clusters=K_CHOSEN, n_init=10, random_state=RANDOM_STATE).fit_predict(X)
complete["path_c_cluster"] = labels_final
print(pd.Series(labels_final).value_counts().sort_index())


In [ ]:
ari_vs_phase1 = adjusted_rand_score(complete["phase1_cluster"], complete["path_c_cluster"])
print(f"ARI(Path C' k={K_CHOSEN}, Phase 1 k=3) = {ari_vs_phase1:.4f}")
print(pd.crosstab(complete["path_c_cluster"], complete["phase1_cluster"]))


In [ ]:
profiles = complete.groupby("path_c_cluster")[FEATURES].mean()
profiles["n"] = complete.groupby("path_c_cluster").size()
print(profiles)


In [ ]:
complete[["participant_id_canonical"] + FEATURES + ["path_c_cluster", "phase1_cluster"]].to_csv(
    OUT_DIR / "path_c_feature_matrix.csv", index=False)
profiles.reset_index().to_csv(OUT_DIR / "path_c_cluster_profiles.csv", index=False)

summary = pd.DataFrame([{
    "n": len(complete), "k_chosen": K_CHOSEN,
    "jaccard": jaccard_by_k[K_CHOSEN], "silhouette": sil_scores[K_CHOSEN],
    "ari_vs_phase1": ari_vs_phase1,
}])
summary.to_csv(OUT_DIR / "path_c_summary.csv", index=False)
print("Wrote path_c_feature_matrix.csv, path_c_cluster_profiles.csv, path_c_summary.csv")


In [ ]:
summary_lines = [
    "# DS4 Step 2, Path C' -- Model Run Summary", "",
    "**Notebook:** `132_ds4_network_focused_phenotype.ipynb`  ",
    "**Plan:** `outputs/reports/stage4/step2_phenotyping/ds4_step2_exploration_plan.md`  ",
    "**Run date:** " + pd.Timestamp.now().strftime("%Y-%m-%d"), "",
    f"Complete-case N={len(complete)}, chosen k={K_CHOSEN} "
    f"(silhouette={sil_scores[K_CHOSEN]:.4f}, jaccard={jaccard_by_k[K_CHOSEN]:.4f})", "",
    f"**ARI vs Phase 1 = {ari_vs_phase1:.4f}**", "",
    "## K-selection (all candidates)", "", "```",
]
for k in k_candidates:
    v = "STABLE" if jaccard_by_k[k] >= 0.75 else "not stable"
    summary_lines.append(f"k={k}: silhouette={sil_scores[k]:.4f}, jaccard={jaccard_by_k[k]:.4f} ({v}), sizes={sizes_by_k[k]}")
summary_lines.append("```")

with open(OUT_DIR / "path_c_model_run_summary.md", "w") as f:
    f.write("\n".join(summary_lines) + "\n")
print("Wrote path_c_model_run_summary.md")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ks = list(sil_scores.keys())
ax.plot(ks, [sil_scores[k] for k in ks], marker="o", color="#55A868")
ax.axvline(K_CHOSEN, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("k")
ax.set_ylabel("silhouette score")
ax.set_title(f"Path C' (network-focused): silhouette by k (chosen k={K_CHOSEN})")
fig.tight_layout()
fig.savefig(FIG_DIR / "path_c_fig1_silhouette_by_k.png", dpi=150)
plt.close(fig)


In [ ]:
ct = pd.crosstab(complete["path_c_cluster"], complete["phase1_cluster"])
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(ct.values, cmap="Greens")
ax.set_xticks(range(ct.shape[1]))
ax.set_xticklabels(ct.columns)
ax.set_yticks(range(ct.shape[0]))
ax.set_yticklabels(ct.index)
for i in range(ct.shape[0]):
    for j in range(ct.shape[1]):
        ax.text(j, i, ct.values[i, j], ha="center", va="center",
                color="white" if ct.values[i, j] > ct.values.max()/2 else "black")
ax.set_xlabel("Phase 1 cluster (k=3, 5 features)")
ax.set_ylabel(f"Path C' cluster (k={K_CHOSEN}, network-focused)")
ax.set_title(f"Path C' vs Phase 1 (ARI={ari_vs_phase1:.3f})")
fig.colorbar(im, ax=ax, label="n participants")
fig.tight_layout()
fig.savefig(FIG_DIR / "path_c_fig2_crosstab_vs_phase1.png", dpi=150)
plt.close(fig)


## Interpretation and limits

**Read the cells above for live numbers.**

- A near-zero ARI vs Phase 1 means network structure forms a genuinely separate phenotype axis that Phase 1's blended 5-feature solution (1 NeT feature diluted among 4 behavioral ones) did not fully capture. Combined with Path A' (high ARI vs Phase 1), the picture is: DS4's dominant axis is behavioral, but network structure is a real, independent secondary axis -- not noise, not redundant, just underweighted in a single blended clustering pass.
- `rho_self`/`I_consec` inherit the same daily-timescale and no-condition caveats as every other DS4 notebook.


## Forward connections

| Connection | Status |
|---|---|
| Path F' (joint taxonomy, A' x C') | Next -- crosses both axes |
